In [1]:
import re 
import numpy as np
import pandas as pd
from datasets import load_dataset, concatenate_datasets
from datasets import Dataset, DatasetDict
from transformers import (BertTokenizer, BertForSequenceClassification, 
                          Trainer, TrainingArguments, AutoTokenizer,
                          DataCollatorWithPadding)

In [2]:
ds = load_dataset("parquet", data_files = "./dataset/wudao/*.parquet", split="train",)
size_need_data = int(len(ds) * 0.05)
ds

Dataset({
    features: ['text'],
    num_rows: 354798
})

In [3]:
def clean_text(text):
    if not text or not isinstance(text, str):
        return False
    
    # 1. 去掉HTML标签（增强版）
    text = re.sub(r'<[^>]+>', '', text)  # 更稳健的HTML标签匹配
    
    # 2. 去掉空文本（增强检查）
    if not text.strip():
        return False
    
    # 3. 去掉各种类型的电话号码
    text = re.sub(r'\b\d{3}[-\.\s]??\d{3}[-\.\s]??\d{4}\b', '', text)  # 带分隔符的电话
    text = re.sub(r'\b\d{11}\b', '', text)  # 11位手机号
    text = re.sub(r'\b\d{4}[-\.\s]??\d{3}[-\.\s]??\d{4}\b', '', text)  # 带区号的电话
    
    # 4. 扩展广告语和推广内容识别
    ad_patterns = [
        '关注公众号', '扫码.*获取', '添加微信', '点击下方链接',
        '详情请访问', '领取优惠券', '限时折扣', '立即购买', '了解更多',
        '欢迎转载', '版权声明', '免责声明', '文章来源', '发布于',
        'tel', '电话', '联系电话', '热线', '企鹅', 'qq', 'q号'
    ]
    for pattern in ad_patterns:
        text = re.sub(pattern, '', text)
    
    # 5. 处理URL和电子邮件
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '', text)
    
    # 6. 扩展表情符号和特殊字符处理
    text = re.sub(r'[\U00010000-\U0010FFFF]', '', text)  # 高位Unicode表情
    text = re.sub(r'[\u2000-\u2FFF]', '', text)  # 扩展标点、符号
    text = re.sub(r'[\u3000-\u303F]', '', text)  # 中文标点符号（可选）
    text = re.sub(r'[\[\]{}<>#*★◇§♡♥♪♬▶▼▪◆●¡⭐]', '', text)  # 更多特殊符号
    
    # 7. 处理重复标点和无意义字符序列
    text = re.sub(r'[!?。，]{2,}', '.', text)  # 多个重复标点替换为单个
    text = re.sub(r'[\.]{2,}', '.', text)  # 多个句点替换为单个
    text = re.sub(r'[\s]+', ' ', text)  # 多个空白字符替换为单个空格
    text = re.sub(r'\s+', ' ', text).strip() # 替换重复的空格
    
    # 8. 处理无意义短文本
    # 如果清洗后文本太短，可能是无意义内容
    cleaned_text = text.strip()
    if len(cleaned_text) < 64:  # 可根据实际调整阈值
        return False
    
    return cleaned_text


In [4]:
for i in range(2):
    ds_clean = ds.filter(lambda x: clean_text(x["text"]))
print(ds_clean)

Dataset({
    features: ['text'],
    num_rows: 354228
})


In [5]:
model_path = "./model/classification/BERT_classifier_for_wudao/checkpoint-6654"
BERT_model = BertForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path) 

In [6]:
ds_test = ds_clean #.select(range(10000))
test = ds_test.add_column("labels", [0] * len(ds_test))
def tokenize(batch):
    return tokenizer(batch["text"], padding=False, truncation=True, max_length=512)

tokened_dataset = test.map(tokenize, batched=True)
tokened_dataset

KeyboardInterrupt: 

In [11]:
test_train_args = TrainingArguments(
    output_dir="./temp_eval",
    per_device_eval_batch_size=8,
    do_train=False,
    do_eval=True,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest")

trainer = Trainer(
    model = BERT_model,
    args=test_train_args,
    processing_class = tokenizer,
    data_collator=data_collator
)

In [ ]:
predictions = trainer.predict(tokened_dataset)
logits = predictions.predictions  # 模型输出的 logits
import numpy as np
pred_labels = np.argmax(logits, axis=1)  # 转换为标签
print(type(pred_labels), len(pred_labels))


C:\Users\hhm18\miniconda3\envs\env_LLM\lib\site-packages\transformers\models\bert\modeling_bert.py:412: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


<class 'numpy.ndarray'> 354228


In [9]:
pred_labels

array([6, 6, 5, ..., 5, 2, 4], dtype=int64)

In [34]:
unique_elements,  counts =np.unique(pred_labels, return_counts=True)
unique_elements, counts

(array([0, 1, 2, 3, 4, 5, 6, 7], dtype=int64),
 array([  6132,  55889,  11931,  50083,  41813, 116248,  41679,  30453],
       dtype=int64))

In [16]:
tokened_dataset, ds_test

(Dataset({
     features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 354228
 }),
 Dataset({
     features: ['text'],
     num_rows: 354228
 }))

In [ ]:
data_labels = ds_test.add_column("labels", pred_labels)

In [ ]:
dataset_list = [
    data_labels.filter(lambda x, i=i: x["labels"] == i)
    for i in range(8)
]
dataset_list

Filter:   0%|          | 0/354228 [00:00<?, ? examples/s]

Filter:   0%|          | 0/354228 [00:00<?, ? examples/s]

Filter:   0%|          | 0/354228 [00:00<?, ? examples/s]

Filter:   0%|          | 0/354228 [00:00<?, ? examples/s]

Filter:   0%|          | 0/354228 [00:00<?, ? examples/s]

Filter:   0%|          | 0/354228 [00:00<?, ? examples/s]

Filter:   0%|          | 0/354228 [00:00<?, ? examples/s]

Filter:   0%|          | 0/354228 [00:00<?, ? examples/s]

[Dataset({
     features: ['text', 'labels'],
     num_rows: 6132
 }),
 Dataset({
     features: ['text', 'labels'],
     num_rows: 55889
 }),
 Dataset({
     features: ['text', 'labels'],
     num_rows: 11931
 }),
 Dataset({
     features: ['text', 'labels'],
     num_rows: 50083
 }),
 Dataset({
     features: ['text', 'labels'],
     num_rows: 41813
 }),
 Dataset({
     features: ['text', 'labels'],
     num_rows: 116248
 }),
 Dataset({
     features: ['text', 'labels'],
     num_rows: 41679
 }),
 Dataset({
     features: ['text', 'labels'],
     num_rows: 30453
 })]

In [ ]:
for k in range(len(counts)):
    print(dataset_list[k][:8])

{'text': ['加味益心汤\n加味益心汤用于补益心气，温脾理痰。主治心气不足，兼有脾湿而致心悸（房颤），头晕，冷汗多，便溏，脉右关沉滑，左沉弱，均有结代，舌苔薄白。法半夏2钱，茯苓2钱，化橘红1钱半，炙甘草5分，炒枣仁3钱，远志1钱，石菖蒲8分，党参1钱半，枳实8分，松节3钱。 补益心气，温脾理痰。主心气不足，兼有脾湿而致心悸（房颤），头晕，冷汗多，便溏，脉右关沉滑，左沉弱，均有结代，舌苔薄白。 每日1剂，水煎服。 方出《蒲辅周医疗经验》，名见《千家妙方》上册', '今天1e ro买了三个半边的茄子,想心思一次做完,就用这个鱼香茄子煲吧\n【美食杰热菜菜谱菜谱大全】鱼香茄子煲的做法 原料:长茄子2只,猪肉馅60克,胡萝卜100克,冬笋100克,木耳2大朵。 调味料:葱末15克,姜末10克,蒜末15克,豆瓣酱1汤匙,麻油12茶匙,生抽、老抽各12茶匙,盐12茶勺,白糖1茶勺。 腌肉料:生粉12茶匙,生抽12茶匙,盐14茶勺,料酒14茶勺,胡椒粉适量。 水淀粉:生粉1茶匙,清水2汤匙。 1、茄子去皮,切成7cm长、2cm见方的长条。 2、猪肉馅加入调味料腌制20分钟,木耳提前用温水泡发后切丝,胡萝卜、冬笋切丝。 3、锅中放半斤油,烧7成热,把茄子放入炸,其间保持中火和翻动,以免炸糊。 4、茄子炸成金黄色、软到筷子可以戳透的时候捞出,捞的时候用勺子压压,可以控出较多的油,或者将煎炸好的茄条在沸水锅中汆一下,以除去多余油分,然后捞出沥干水分。 5、炒锅放少许油,加热到7成热,放入蒜末煸炒出香味,然后加入葱末、姜末和豆瓣酱继续煸炒。 6、加入腌好的肉馅,煸炒熟,然后放入笋丝、胡萝卜丝和木耳丝,煸炒。 7、把笋丝、胡萝卜丝和木耳丝炒熟后,放入炸好的茄子条,调入生抽、老抽、盐、白糖和高汤。 8、大火烧至茄条入味且全熟时,调入鸡精,随后用水淀粉勾薄芡,淋入香油 9、盛进事先烧烫的小砂锅(煲仔)内。淋上辣油后加砂锅盖,小火焖5分钟,撒葱花趁热上桌即可食用。', '山西【大同黄糕】\n大同黄糕是山西大同一带常见的汉族糕类家常食品,原料是黄米面,先用温水和成碎块状(散粒),上笼蒸熟,然后倒在盆里用手再揉一遍,边揉边在其表面抹点麻油,这样作可以防止糕面表皮干裂。最后把和好的糕面分成小块,蘸上肉菜汁即可食用。黄糕具有"黄、软、筋、香"四大特点,吃起来松软可口,十分味美。 黄糕,是用黄米

In [ ]:
for k in range(8):
    print(f"labels_l{k}" , counts[k])

labels_l0 6132
labels_l1 55889
labels_l2 11931
labels_l3 50083
labels_l4 41813
labels_l5 116248
labels_l6 41679
labels_l7 30453


In [ ]:
persent = [0.6, 0.95, 0.6, 0.85, 0.1, 0.75, 0.3 ,0.8 ]
selected_data = np.sum(counts * persent)
print("现有数据", selected_data, "\n占比:", selected_data / counts.sum())

现有数据 234736.3 
占比: 0.6626700881917861


In [ ]:
assert len(persent) == len(counts)
selected_datasets = []

for i, dataset in enumerate(dataset_list):
    n_sample = int(len(dataset) * persent[i])
    n_sample = max(1, n_sample) if persent[i] > 0 else 0

    if n_sample ==0:
        continue
    
    selected_indices = np.random.choice(len(dataset), n_sample, replace=False)
    selected_dataset = dataset.select(selected_indices)
    selected_datasets.append(selected_dataset)
    print(f"从数据集 {i+1} 中抽取了 {n_sample} 个样本 (权重: {persent[i]})")

concat_dataset = concatenate_datasets(selected_datasets)
print(f"最终拼接的数据集包含 {len(concat_dataset)} 个样本")

从数据集 1 中抽取了 3679 个样本 (权重: 0.6)
从数据集 2 中抽取了 53094 个样本 (权重: 0.95)
从数据集 3 中抽取了 7158 个样本 (权重: 0.6)
从数据集 4 中抽取了 42570 个样本 (权重: 0.85)
从数据集 5 中抽取了 4181 个样本 (权重: 0.1)
从数据集 6 中抽取了 87186 个样本 (权重: 0.75)
从数据集 7 中抽取了 12503 个样本 (权重: 0.3)
从数据集 8 中抽取了 24362 个样本 (权重: 0.8)
最终拼接的数据集包含 234733 个样本


In [ ]:
concat_dataset.to_parquet("./dataset/wudao/clean_weight.parquet")

Creating parquet from Arrow format:   0%|          | 0/235 [00:00<?, ?ba/s]

801953706